# H1-v2 Tiny16 assignment and normalization diagnostic

This notebook performs no training. It reuses the five saved Tiny16 checkpoints to measure GT-to-query assignment churn and compare ordinary eval-mode BatchNorm running statistics against batch-statistics inference. Run all cells on a Colab GPU and return the final JSON report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import json, os, shlex, subprocess, sys
REPO_URL='https://github.com/Ali-RT/mobile_adas3d.git'; BRANCH='main'
PROJECT_DIR=Path('/content/mobile_adas3d')
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti'); DATASET_VIEW=Path('/content/kitti_h1')
TINY_SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/h1_v2_tiny16')
OUTPUT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_outputs/mobileadas3d_h1_v2_tiny16_2000step')
CONFIG=PROJECT_DIR/'configs/kitti_mobileadas3d_h1_v2_tiny_2000step.yaml'
RUN_NAME='mobileadas3d_h1_v2_tiny16_2000step'
def run(command,cwd=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    result=subprocess.run(command,cwd=cwd)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_streamed(command,cwd,log_path):
    command=[str(x) for x in command]; log_path.parent.mkdir(parents=True,exist_ok=True)
    print('+',shlex.join(command),flush=True); print('Durable log:',log_path,flush=True)
    tail=deque(maxlen=160); env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=p.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))


In [ ]:
# Refresh code and require CUDA. The diagnostic commit must be available remotely first.
if not (PROJECT_DIR/'.git').exists(): run(['git','clone','--branch',BRANCH,REPO_URL,PROJECT_DIR])
else:
    run(['git','fetch','origin'],PROJECT_DIR); run(['git','checkout',BRANCH],PROJECT_DIR); run(['git','pull','--ff-only','origin',BRANCH],PROJECT_DIR)
os.chdir(PROJECT_DIR)
run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],PROJECT_DIR)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Choose Runtime > Change runtime type > GPU')
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_DIR,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
# Resolve the existing complete KITTI view; no dataset copying is required.
def count_files(path,suffix): return sum(1 for p in path.iterdir() if p.is_file() and p.suffix==suffix) if path.is_dir() else 0
def complete(root): return count_files(root/'training/image_2','.png')==7481 and count_files(root/'training/label_2','.txt')==7481 and count_files(root/'training/calib','.txt')==7481
if complete(LOCAL_DATASET_ROOT): DATASET_ROOT=LOCAL_DATASET_ROOT
else:
    aliases={'image_2':['image_2','image_02'],'label_2':['label_2','label_02'],'calib':['calib']}
    (DATASET_VIEW/'training').mkdir(parents=True,exist_ok=True)
    for canonical,candidates in aliases.items():
        source=next((DRIVE_DATASET_ROOT/'training'/name for name in candidates if (DRIVE_DATASET_ROOT/'training'/name).is_dir()),None)
        if source is None: raise FileNotFoundError(f'Missing source for {canonical}')
        link=DATASET_VIEW/'training'/canonical
        if not link.exists() and not link.is_symlink(): link.symlink_to(source,target_is_directory=True)
    DATASET_ROOT=DATASET_VIEW
if not complete(DATASET_ROOT): raise RuntimeError(f'KITTI view incomplete: {DATASET_ROOT}')
for name in ('train.txt','val.txt'):
    path=TINY_SPLIT_DIR/name
    ids=[x for x in path.read_text().splitlines() if x.strip()] if path.is_file() else []
    if len(ids)!=16: raise RuntimeError(f'Expected existing Tiny16 {name}, got {len(ids)}')
print('Dataset root:',DATASET_ROOT); print('Tiny split:',TINY_SPLIT_DIR)


In [ ]:
# Locate exactly one completed 2,000-step run and verify all five milestone checkpoints.
runs=[]
for latest in OUTPUT_DIR.glob(f'runs/*{RUN_NAME}*/checkpoints/latest.pt'):
    state=torch.load(latest,map_location='cpu',weights_only=False)
    if state.get('epoch')==500 and state.get('global_step')==2000: runs.append(latest.parent.parent)
if len(runs)!=1: raise RuntimeError(f'Expected exactly one completed run, found {len(runs)}: {runs}')
TRAIN_RUN_DIR=runs[0]
CHECKPOINTS=[(400,TRAIN_RUN_DIR/'checkpoints/epoch_100.pt'),(800,TRAIN_RUN_DIR/'checkpoints/epoch_200.pt'),(1200,TRAIN_RUN_DIR/'checkpoints/epoch_300.pt'),(1600,TRAIN_RUN_DIR/'checkpoints/epoch_400.pt'),(2000,TRAIN_RUN_DIR/'checkpoints/epoch_500.pt')]
missing=[str(path) for _,path in CHECKPOINTS if not path.is_file()]
if missing: raise FileNotFoundError('Missing milestones:\n'+'\n'.join(missing))
print('Run:',TRAIN_RUN_DIR)
for step,path in CHECKPOINTS: print(step,path)


In [ ]:
# Run assignment-churn and BatchNorm-mode diagnostics. This performs no optimization.
REPORT=TRAIN_RUN_DIR/'h1_v2_assignment_normalization_diagnostic.json'
command=[sys.executable,'-u','scripts/diagnose_h1_assignment_normalization.py','--config',CONFIG,'--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',TINY_SPLIT_DIR,'--output-dir',OUTPUT_DIR,'--split','val','--score-threshold','0.1','--report',REPORT]
for step,path in CHECKPOINTS: command += ['--checkpoint',f'{step}={path}']
run_streamed(command,PROJECT_DIR,OUTPUT_DIR/'colab_logs'/'assignment_normalization_diagnostic.log')
report=json.loads(REPORT.read_text())
print(json.dumps({
    'report':str(REPORT),
    'assignment_stability':{k:v for k,v in report['eval_assignment_stability'].items() if k!='per_object'},
    'mode_comparison':[{'step':x['step'],'eval':x['modes']['eval'],'batch_stats':x['modes']['batch_stats']} for x in report['checkpoints']]
},indent=2))
print('Send this file for review:',REPORT)
